In [ ]:
import os
from datetime import datetime

from dotenv import load_dotenv
from pydantic import BaseModel

from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_tavily import TavilySearch


# ============================================================
# CONFIGURATION
# ============================================================

load_dotenv()
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
TAVILY_API_KEY = os.environ.get("TAVILY_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is missing.")

if not TAVILY_API_KEY:
    raise ValueError("TAVILY_API_KEY is missing.")


# ============================================================
# STRUCTURED OUTPUT
# ============================================================

class AgentResponse(BaseModel):
    answer: str
    success: bool


# ============================================================
# MODEL
# ============================================================

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    groq_api_key=GROQ_API_KEY
)


# ============================================================
# TAVILY
# ============================================================

tavily = TavilySearch(
    max_result=5,
    tavily_api_key=TAVILY_API_KEY
)


# ============================================================
# TOOLS
# ============================================================

@tool
def calculate(
    a: float,
    b: float,
    operation_type: str
):
    """
    Performs basic arithmetic using two numbers and one operation.

    Supported operations:
    addition (+)
    subtraction (-)
    multiplication (*)
    division (/)

    Division by zero is not allowed.
    """

    operation_type = operation_type.strip().lower()

    if operation_type == "+":
        return a + b

    elif operation_type == "-":
        return a - b

    elif operation_type == "*":
        return a * b

    elif operation_type == "/":

        if b == 0:
            raise ValueError(
                "Cannot divide by zero."
            )

        return a / b

    else:
        raise ValueError(
            f"Unsupported operation: {operation_type}. "
            "Supported operations are +, -, *, /."
        )


@tool
def get_time():
    """
    Get the current date and time.
    """

    return datetime.now().isoformat()


@tool
def web_search(query: str):
    """
    Search the web for current information.
    """

    if not query.strip():
        raise ValueError(
            "Search query cannot be empty."
        )

    try:

        result = tavily.invoke(
            {
                "query": query
            }
        )

        return str(result)

    except Exception as e:

        raise RuntimeError(
            f"Web search failed: {str(e)}"
        )


# ============================================================
# AGENT
# ============================================================

agent = create_agent(
    model=llm,
    tools=[
        calculate,
        get_time,
        web_search
    ],
    system_prompt="""
You are a research assistant and tutor for school-going kids.

Keep answers short and simple.

You have access to three tools:

1. calculate
   Use this for arithmetic.

2. get_time
   Use this when the user asks for the current date or time.

3. web_search
   Use this when the user needs current or external information.

Rules:

- Use tools when necessary.
- You can call multiple tools for one question.
- You can call the same tool multiple times if necessary.
- Carefully inspect the result of a tool before deciding what to do next.
- If a tool fails, try another approach when possible.
- Do not pretend a failed tool call succeeded.
- When you have enough information, provide the final answer.
"""
)


# ============================================================
# DEBUGGING / INSPECTION
# ============================================================

def inspect_message(message):
    """
    Inspect what happened at each step of the agent loop.
    """

    message_type = message.__class__.__name__

    print("\n----------------------------------------")
    print(f"MESSAGE TYPE: {message_type}")
    print("----------------------------------------")

    # --------------------------------------------------------
    # Message content
    # --------------------------------------------------------

    if hasattr(message, "content"):

        if message.content:
            print("\nCONTENT:")
            print(message.content)


    # --------------------------------------------------------
    # Tool calls
    # --------------------------------------------------------

    if hasattr(message, "tool_calls"):

        if message.tool_calls:

            print("\nTOOL CALL:")

            for tool_call in message.tool_calls:

                print(
                    f"Tool: {tool_call.get('name')}"
                )

                print(
                    f"Arguments: {tool_call.get('args')}"
                )

                print(
                    f"ID: {tool_call.get('id')}"
                )


    # --------------------------------------------------------
    # Tool result
    # --------------------------------------------------------

    if message_type == "ToolMessage":

        print("\nTOOL RESULT:")

        print(message.content)


# ============================================================
# RUN AGENT
# ============================================================

def run_agent(user_input, conversation_history):

    print("\n")
    print("=" * 60)
    print("USER")
    print("=" * 60)

    print(user_input)

    print("\n")
    print("=" * 60)
    print("AGENT TRACE")
    print("=" * 60)


    try:

        response = agent.stream(
            {
                "messages": conversation_history + [
                    {
                        "role": "user",
                        "content": user_input
                    }
                ]
            },

            stream_mode="values"
        )


        final_state = None


        # ====================================================
        # AGENT LOOP
        # ====================================================

        for chunk in response:

            final_state = chunk

            messages = chunk.get(
                "messages",
                []
            )

            if not messages:
                continue


            latest_message = messages[-1]


            # ------------------------------------------------
            # Inspect current step
            # ------------------------------------------------

            inspect_message(
                latest_message
            )


            # ------------------------------------------------
            # Detect tool call
            # ------------------------------------------------

            if hasattr(
                latest_message,
                "tool_calls"
            ):

                if latest_message.tool_calls:

                    print(
                        "\n>>> AGENT DECISION: CALL TOOL"
                    )

                    for tool_call in latest_message.tool_calls:

                        print(
                            f"Tool selected: "
                            f"{tool_call.get('name')}"
                        )

                        print(
                            f"Arguments: "
                            f"{tool_call.get('args')}"
                        )


            # ------------------------------------------------
            # Detect tool result
            # ------------------------------------------------

            if (
                latest_message.__class__.__name__
                == "ToolMessage"
            ):

                print(
                    "\n>>> TOOL EXECUTED"
                )


        # ====================================================
        # FINAL STATE
        # ====================================================

        if final_state is None:

            print(
                "\nAgent returned no final state."
            )

            return conversation_history


        final_messages = final_state.get(
            "messages",
            []
        )


        if not final_messages:

            print(
                "\nNo messages returned."
            )

            return conversation_history


        # ====================================================
        # FINAL ANSWER
        # ====================================================

        print("\n")
        print("=" * 60)
        print("FINAL ANSWER")
        print("=" * 60)


        last_message = final_messages[-1]


        print(
            last_message.content
        )


        # ====================================================
        # STRUCTURED RESPONSE
        # ====================================================

        structured_response = final_state.get(
            "structured_response"
        )


        if structured_response:

            print("\n")
            print("=" * 60)
            print("STRUCTURED RESPONSE")
            print("=" * 60)

            print(
                structured_response
            )


        # ====================================================
        # RETURN CONVERSATION
        # ====================================================

        return final_messages


    except Exception as e:

        print("\n")
        print("=" * 60)
        print("AGENT ERROR")
        print("=" * 60)

        print(
            f"{type(e).__name__}: {e}"
        )

        print(
            "\nThe agent failed to complete this request."
        )

        return conversation_history


# ============================================================
# MAIN LOOP
# ============================================================

conversation_history = []


while True:

    user_input = input(
        "\nEnter your question... "
    ).strip()


    # --------------------------------------------------------
    # Empty input
    # --------------------------------------------------------

    if not user_input:

        print(
            "Please enter a question."
        )

        continue


    # --------------------------------------------------------
    # Exit
    # --------------------------------------------------------

    if user_input.lower() in [
        "stop",
        "exit",
        "break",
        "quit"
    ]:

        print(
            "\nGoodbye!"
        )

        break


    # --------------------------------------------------------
    # Run agent
    # --------------------------------------------------------

    conversation_history = run_agent(
        user_input,
        conversation_history
    )




USER
what is gravity?


AGENT TRACE

----------------------------------------
MESSAGE TYPE: HumanMessage
----------------------------------------

CONTENT:
what is gravity?


AGENT ERROR
BadRequestError: Error code: 400 - {'error': {'message': 'json mode cannot be combined with tool/function calling', 'type': 'invalid_request_error', 'param': 'response_format'}}

The agent failed to complete this request.
